In [3]:
#########################
# Include Headers
#########################

using LinearAlgebra
using HDF5
using Arpack
using CairoMakie


include("../header/.src/circuit/brickwall.jl")
include("../header/.src/circuit/heisenberg.jl")
#include("../header/.src/circuit/time-crystals.jl")
include("../header/.src/hdf5/hdf5Mods.jl")
include("../header/.src/plotmods/colorschemes_mods.jl")
include("../header/.src/random_matrices/random_matrices.jl")
include("../header/.src/measurements/projectors.jl")
include("../header/.src/info_lattice/info_lattice.jl")
include("../header/.src/lazadires_diagram/lazadires_diagram.jl")


LazadiresDiagramPlot (generic function with 1 method)

In [4]:
#### attrs-author-and-generating-file for given L

L=8

file=h5open("../data/hiesenberg_transition_ordpar_L$(L).hdf5","cw")

set_hdf5_attributes(file)

close(file);

┌ Warning: Pkg.installed() is deprecated
└ @ Pkg C:\Users\Ritam\.julia\juliaup\julia-1.11.7+0.x64.w64.mingw32\share\julia\stdlib\v1.11\Pkg\src\Pkg.jl:785


In [ ]:
#### attrs-model

file=h5open("../data/hiesenberg_transition_ordpar.hdf5","cw")
 attrs = HDF5.attributes(file)

 attrs["[Model] 1. Unitary"]="exp(- i J_i ZZ + \theta/2 (XX+YY))exp(- i h_i Z)"

 attrs["[Model] 2. Tuning parameter"]="\theta"
 
 attrs["[Model] 3. Range of J"]="[0, pi], Uniform Sampling"

 attrs["[Model] 4. Range of h"]="[0, 2pi], Uniform Sampling"

 attrs["[Model] 5. Order parameters"]="eigenvalues, info_lattice"

 attrs["4. METADATA"]=["This file contains raw eigen data and information lattice of individual eigenstates of the XXZ unitary, we aim to establish the critical point of the ETH-MBL transition with the aim to establish the unitary as a candidate driving scheme for the fisher information based probe for measurment induced phase transition as proposed by Arnau, Silvia and Xhek.
 
 We save the realistion of h, J for each realisation and since this is the only source of random-ness, results can be exactly reproduced. Relevant functions used to generate the circuit will be provided as a script. info-lattice Data for various projected eigenstates to be added in a seperate file. We also record benchmark times for the Arpack.eigen() and computation of info_lattice seperately. The phase-space scan is run after running a pre-compilation. benchmarktime for pre-compilation is also provided for reference."]



close(file)


In [ ]:
## towards the iterative script for collecting eigen-data

L=8
N=2^L


thetalist=collect(1.0:0.1:4.5)


for theta in thetalist

        for itr in 1:30
        
        file=h5open("../data/hiesenberg_transition_ordpar_$(file_no).hdf5","cw")

        ## Drawing the Background Disorder

        h=rand(L)*2*pi;


        ## Writing the disorder strength

        file["L$(L)/theta$(theta)/Itr$(itr)/h"]=h;

        
        ## Building Ckt, Getting Eigenstates

        A=circuit_heisenberg(L,theta,h)
        eigvals,eigvecs= eigen(A)


        ## Writing eigenvalues

        file["L$(L)/theta$(theta)/Itr$(itr)/eigvals"]=eigvals;



        ## Writing info-lattice of individual eigenstates


        for i in 1:N
                state=eigvecs[i,:]
                info_lattice_state=info_lattice(state)
                file["L$(L)/theta$(theta)/Itr$(itr)/info_lattice/eigvec_$(i)"]=lattice_to_vec(info_lattice(state))
        end

        close(file)

        end
        print("$(theta) \n")
        GC.gc()
end


